# 🧪 Semana 14 · Unidad 4 — Laboratorio: Binary Search Trees (BST)

**Universidad de Talca — Curso de Algoritmos y Estructuras de Datos**

| Aspecto | Detalle |
|--------|--------|
| **Profesor** | PhD. César Astudillo |
| **Unidad** | Unidad 4: Diccionarios |
| **Tema** | Binary Search Trees — Implementación y Operaciones de Orden |
| **Duración** | 100 minutos (2 bloques de 50 min) |

---

## 📋 Instrucciones Generales

- Trabaja de forma **individual**.
- Ejecuta cada celda antes de pasar a la siguiente.
- Las pruebas automáticas usan `unittest`, la librería estándar de Python (ver S00 · Unit Testing): cada prueba termina en `ok`, `FAIL` (resultado incorrecto) o `ERROR` (tu código lanzó una excepción). Cada prueba aprobada vale 1 punto (las marcadas como «2 pts» valen 2).
- Al terminar el Bloque 1, el ayudante revisará tu avance antes de continuar.
- **No modifiques** las celdas de Setup ni las de verificación automática.

In [ ]:
## Setup (NO MODIFICAR)
import math
import random
import unittest

PUNTAJE = {}     # parte → (puntos obtenidos, puntos posibles)

def correr_pruebas(parte, caso, pesos=None):
    """
    Corre con unittest las pruebas de la clase `caso` y anota el puntaje de la parte.

    Cada método test_ vale 1 punto, salvo los indicados en `pesos` ({nombre: puntos}).
    Una prueba que falla en varios subTest se descuenta una sola vez.
    """
    pesos = pesos or {}
    programa = unittest.main(argv=["ignorado", caso.__name__], exit=False, verbosity=2)
    r = programa.result
    fallidas = {getattr(t, "test_case", t).id().split(".")[-1] for t, _ in r.failures + r.errors}
    nombres = unittest.TestLoader().getTestCaseNames(caso)
    posibles = sum(pesos.get(m, 1) for m in nombres)
    obtenidos = sum(pesos.get(m, 1) for m in nombres if m not in fallidas)
    PUNTAJE[parte] = (obtenidos, posibles)
    print(f"\n  Subtotal {parte}: {obtenidos}/{posibles} puntos")

def _size(node):
    return 0 if node is None else node.size

def altura_bst(root):
    if root is None: return 0
    return 1 + max(altura_bst(root.left), altura_bst(root.right))

def size_correcto(node):
    """True si en todo el subárbol se cumple size = 1 + size(izq) + size(der)."""
    if node is None:
        return True
    return (node.size == 1 + _size(node.left) + _size(node.right)
            and size_correcto(node.left) and size_correcto(node.right))

def dibujar_bst(root, titulo="BST"):
    """
    Imprime un BST al estilo del comando `tree`.

    Cada hijo indica si es izquierdo (izq) o derecho (der) y, entre paréntesis,
    el campo size de su subárbol.
    """
    print(titulo)
    if root is None:
        print("Árbol vacío")
        return
    print(f"{root.key} (sz={root.size})")

    def hijos(node, prefijo):
        presentes = [(lado, h) for lado, h in (("izq", node.left), ("der", node.right)) if h is not None]
        for i, (lado, h) in enumerate(presentes):
            ultimo = i == len(presentes) - 1
            print(f"{prefijo}{'└── ' if ultimo else '├── '}{lado}: {h.key} (sz={h.size})")
            hijos(h, prefijo + ("    " if ultimo else "│   "))

    hijos(root, "")
    print()

print("✅ Setup listo")

---
# 🔵 BLOQUE 1 — Estructura BST: get y put (50 minutos)

> Al terminar este bloque, levanta la mano para que el ayudante revise tu progreso.

## PARTE 1A: Trazar get y put a mano (12 minutos)

Considera el siguiente BST construido insertando en este orden:  
`'S', 'E', 'X', 'A', 'R', 'H', 'M'`

```
          S
         / \
        E   X
       / \
      A   R
         /
        H
         \
          M
```

### Pregunta 1 — traza de `get('M')`

Completa la tabla mostrando cada paso de la búsqueda:

| Paso | Nodo actual | Comparación | Dirección |
|------|------------|-------------|----------|
| 1 | S | 'M' < 'S' | Izquierda |
| 2 | ? | ? | ? |
| 3 | ? | ? | ? |
| 4 | ? | ? | ? |
| 5 | ? | ? | ENCONTRADO |

**¿Cuántos nodos se visitaron?** ___

### Pregunta 2 — traza de `put('C', 3)`

Sigue el camino que recorrería `put('C', 3)` hasta encontrar dónde insertar la nueva hoja.

| Paso | Nodo actual | Comparación | Dirección |
|------|------------|-------------|----------|
| 1 | S | 'C' < 'S' | Izquierda |
| 2 | ? | ? | ? |
| 3 | ? | ? | ? |
| 4 | ? | ? | Insertar aquí |

**¿Cómo queda el árbol (dibuja o describe)?** ___

### Pregunta 3 — campo size

Tras insertar `'C'`, ¿qué nodos ven modificado su campo `size`?  
Lista los nodos y sus nuevos valores de `size`.

**Respuesta:** ___

### Tu Respuesta 1A (edita esta celda)

**Pregunta 1 — `get('M')`:**

| Paso | Nodo actual | Comparación | Dirección |
|------|------------|-------------|----------|
| 1 | S | 'M' < 'S' | Izquierda |
| 2 | ... | ... | ... |
| 3 | ... | ... | ... |
| 4 | ... | ... | ... |
| 5 | ... | ... | ENCONTRADO |

Nodos visitados: ___

**Pregunta 2 — `put('C', 3)`:**

| Paso | Nodo actual | Comparación | Dirección |
|------|------------|-------------|----------|
| 1 | S | 'C' < 'S' | Izquierda |
| 2 | ... | ... | ... |
| 3 | ... | ... | ... |
| 4 | ... | ... | Insertar aquí |

Árbol resultante: ___

**Pregunta 3 — campo size:** ___

## PARTE 1B: Implementar Node y BST básico (20 minutos)

Implementa la clase `Node` y el BST con `get`, `put`, `size`, `contains` y `keys` (inorden).

In [ ]:
# TODO: Implementa Node
# REQUISITOS:
#   - key, value, left, right, size
#   - size inicializa en 1 (el nodo mismo)

class Node:
    def __init__(self, key, value):
        # TU CÓDIGO AQUÍ
        pass

In [ ]:
# TODO: Implementa BST
# REQUISITOS:
#   - _root: referencia a la raíz (None si vacío)
#   - size(): retornar tamaño total del árbol (usa _size del setup)
#   - isEmpty(): True si _root is None
#   - get(key): búsqueda recursiva — O(altura)
#   - put(key, value): inserción recursiva retornando el nodo — O(altura)
#                     actualizar node.size en cada nodo del camino
#   - contains(key): usar get
#   - keys(): inorden (izquierda → raíz → derecha) como lista

class BST:

    def __init__(self):
        self._root = None

    def size(self):
        # TU CÓDIGO AQUÍ
        pass

    def isEmpty(self):
        # TU CÓDIGO AQUÍ
        pass

    def get(self, key):
        """
        Retornar el valor de key, o None si no existe.
        Lanzar ValueError si key es None.
        """
        # TU CÓDIGO AQUÍ
        pass

    def _get(self, node, key):
        """
        Búsqueda recursiva.
        HINT:
          si node es None → retornar None
          si key < node.key → buscar en node.left
          si key > node.key → buscar en node.right
          si key == node.key → retornar node.value
        """
        # TU CÓDIGO AQUÍ
        pass

    def contains(self, key):
        # TU CÓDIGO AQUÍ
        pass

    def put(self, key, value):
        """
        Insertar o actualizar.
        Lanzar ValueError si key es None.
        """
        # TU CÓDIGO AQUÍ
        pass

    def _put(self, node, key, value):
        """
        Inserción recursiva. DEBE retornar el nodo.
        HINT:
          si node es None → retornar Node(key, value)
          si key < node.key → node.left  = _put(node.left,  key, value)
          si key > node.key → node.right = _put(node.right, key, value)
          si key == node.key → node.value = value
          actualizar: node.size = 1 + _size(node.left) + _size(node.right)
          retornar node
        """
        # TU CÓDIGO AQUÍ
        pass

    def keys(self):
        """Lista de claves en orden (inorden)."""
        result = []
        self._inorden(self._root, result)
        return result

    def _inorden(self, node, result):
        """
        Recorrido inorden: izquierda → raíz → derecha.
        """
        # TU CÓDIGO AQUÍ
        pass

In [ ]:
# ══════════════════════════════════════════
# VERIFICACIÓN AUTOMÁTICA — NO MODIFICAR
# ══════════════════════════════════════════

class TestBST(unittest.TestCase):
    """BST básico: get, put, size, contains y keys en inorden."""

    CLAVES = [('S', 0), ('E', 1), ('X', 7), ('A', 8), ('R', 3), ('H', 5), ('M', 9)]

    def arbol(self):
        bst = BST()
        for k, v in self.CLAVES:
            bst.put(k, v)
        return bst

    def test_01_nuevo_vacio(self):
        """isEmpty() es True en un árbol nuevo"""
        self.assertIs(BST().isEmpty(), True)

    def test_02_nuevo_size_cero(self):
        """size() es 0 en un árbol nuevo"""
        self.assertEqual(BST().size(), 0)

    def test_03_get_en_vacio(self):
        """get('A') en un árbol vacío retorna None"""
        self.assertIsNone(BST().get('A'))

    def test_04_size_una_insercion(self):
        """size() es 1 tras insertar la raíz"""
        bst = BST()
        bst.put('S', 0)
        self.assertEqual(bst.size(), 1)

    def test_05_get_raiz(self):
        """get('S') retorna el valor de la raíz"""
        bst = BST()
        bst.put('S', 0)
        self.assertEqual(bst.get('S'), 0)

    def test_06_no_vacio(self):
        """isEmpty() es False tras insertar"""
        bst = BST()
        bst.put('S', 0)
        self.assertIs(bst.isEmpty(), False)

    def test_07_size_siete(self):
        """size() es 7 tras insertar S E X A R H M"""
        self.assertEqual(self.arbol().size(), 7)

    def test_08_get_H(self):
        """get('H') = 5"""
        self.assertEqual(self.arbol().get('H'), 5)

    def test_09_get_A(self):
        """get('A') = 8"""
        self.assertEqual(self.arbol().get('A'), 8)

    def test_10_get_ausente(self):
        """get('Z') = None"""
        self.assertIsNone(self.arbol().get('Z'))

    def test_11_contains_true(self):
        """contains('R') es True"""
        self.assertIs(self.arbol().contains('R'), True)

    def test_12_contains_false(self):
        """contains('Z') es False"""
        self.assertIs(self.arbol().contains('Z'), False)

    def test_13_put_actualiza(self):
        """put('H', 99) actualiza el valor"""
        bst = self.arbol()
        bst.put('H', 99)
        self.assertEqual(bst.get('H'), 99)

    def test_14_actualizar_size_estable(self):
        """actualizar una clave no cambia size()"""
        bst = self.arbol()
        bst.put('H', 99)
        self.assertEqual(bst.size(), 7)

    def test_15_keys_inorden(self):
        """keys() recorre en inorden: claves ascendentes"""
        self.assertEqual(self.arbol().keys(), ['A', 'E', 'H', 'M', 'R', 'S', 'X'])

    def test_16_size_de_la_raiz(self):
        """el size de la raíz coincide con size()"""
        bst = self.arbol()
        self.assertEqual(_size(bst._root), bst.size())

    def test_17_invariante_size(self):
        """todo nodo cumple size = 1 + size(izq) + size(der)"""
        bst = self.arbol()
        bst.put('H', 99)
        self.assertTrue(size_correcto(bst._root))

    def test_18_cincuenta_recuperables(self):
        """50 elementos aleatorios se recuperan todos"""
        bst2 = BST()
        datos = random.sample(range(500), 50)
        for v in datos:
            bst2.put(v, v * 2)
        self.assertTrue(all(bst2.get(v) == v * 2 for v in datos))

    def test_19_cincuenta_en_orden(self):
        """(2 pts) con 50 elementos aleatorios, keys() queda ordenado"""
        bst2 = BST()
        datos = random.sample(range(500), 50)
        for v in datos:
            bst2.put(v, v * 2)
        self.assertEqual(bst2.keys(), sorted(datos))


correr_pruebas("1B", TestBST, pesos={"test_19_cincuenta_en_orden": 2})

## PARTE 1C: Visualización del árbol (5 minutos)

Una vez que pases los tests, visualiza tu árbol para verificar visualmente la propiedad BST.

In [ ]:
# Visualiza tu BST — ejecuta esta celda después de pasar los tests de 1B
bst_viz = BST()
for k, v in [('S',0),('E',1),('X',7),('A',8),('R',3),('H',5),('M',9)]:
    bst_viz.put(k, v)

print(f"Árbol: inorden = {bst_viz.keys()}")
print(f"Altura: {altura_bst(bst_viz._root)}")
dibujar_bst(bst_viz._root, "Mi BST — verificar propiedad BST visualmente")

# Ahora inserta 'C' y observa dónde aparece
bst_viz.put('C', 4)
print(f"\nTras put('C',4): inorden = {bst_viz.keys()}")
dibujar_bst(bst_viz._root, "BST tras insertar 'C'")

---
# 🟠 BLOQUE 2 — Operaciones de Orden y Análisis (50 minutos)

> Al terminar este bloque, el ayudante revisará tu progreso final.

## PARTE 2A: floor y rank a mano (10 minutos)

Dado el siguiente BST:

```
          S (size=7)
         / \
        E   X (size=1)
       / \
      A   R (size=3)
           /
           H (size=2)
            \
             M (size=1)
```

### Pregunta 1 — traza de `floor('G')`

Recuerda el algoritmo:
- Si `node is None` → `None`
- Si `key == node.key` → `node`
- Si `key < node.key` → `floor(node.left, key)`
- Si `key > node.key` → `t = floor(node.right, key)` → retornar `t` si no es None, si no `node`

| Llamada | Nodo | key vs node.key | Retorna |
|---------|------|----------------|--------|
| floor(S, 'G') | S | 'G' < 'S' | floor(E, 'G') |
| floor(E, 'G') | E | 'G' > 'E' | t = floor(R, 'G'), luego... |
| floor(R, 'G') | ? | ? | ? |
| floor(H, 'G') | ? | ? | ? |

**Resultado final de `floor('G')`:** ___

### Pregunta 2 — traza de `rank('H')`

Recuerda:
- Si `key < node.key` → `rank(node.left, key)`
- Si `key > node.key` → `1 + size(node.left) + rank(node.right, key)`
- Si `key == node.key` → `size(node.left)`

| Llamada | Nodo | key vs node.key | Retorna |
|---------|------|----------------|--------|
| rank(S, 'H') | S (size=7) | 'H' < 'S' | rank(E, 'H') |
| rank(E, 'H') | E | 'H' > 'E' | 1 + size(A) + rank(R, 'H') |
| rank(R, 'H') | ? | ? | ? |
| rank(H, 'H') | ? | ? | size(?) = ? |

**Resultado final de `rank('H')`:** ___  
**Significado:** hay ___ claves menores que 'H' en el árbol

### Tu Respuesta 2A (edita esta celda)

**Pregunta 1 — `floor('G')`:**

| Llamada | Nodo | key vs node.key | Retorna |
|---------|------|----------------|--------|
| floor(S, 'G') | S | 'G' < 'S' | floor(E, 'G') |
| floor(E, 'G') | E | 'G' > 'E' | t = floor(R, 'G'), luego... |
| floor(R, 'G') | ... | ... | ... |
| floor(H, 'G') | ... | ... | ... |

Resultado: ___

**Pregunta 2 — `rank('H')`:**

| Llamada | Nodo | key vs node.key | Retorna |
|---------|------|----------------|--------|
| rank(S, 'H') | S | 'H' < 'S' | rank(E, 'H') |
| rank(E, 'H') | E | 'H' > 'E' | 1 + size(A) + rank(R, 'H') |
| rank(R, 'H') | ... | ... | ... |
| rank(H, 'H') | ... | ... | size(?) = ? |

Resultado: ___  
Significado: ___

## PARTE 2B: Implementar operaciones de orden (25 minutos)

Extiende tu `BST` con `min`, `max`, `floor`, `ceiling`, `rank` y `select`.

In [ ]:
# TODO: Implementa OrderedBST extendiendo tu BST

class OrderedBST(BST):
    """
    BST con operaciones de orden completas.
    """

    def min(self):
        """Clave mínima — ir siempre a la izquierda."""
        if self.isEmpty(): raise IndexError("Árbol vacío")
        # TU CÓDIGO AQUÍ
        pass

    def _min(self, node):
        """
        Retornar el nodo con la clave mínima del subárbol.
        HINT: si node.left es None → este nodo es el mínimo
              si no → _min(node.left)
        """
        # TU CÓDIGO AQUÍ
        pass

    def max(self):
        """Clave máxima — ir siempre a la derecha."""
        if self.isEmpty(): raise IndexError("Árbol vacío")
        # TU CÓDIGO AQUÍ
        pass

    def _max(self, node):
        # TU CÓDIGO AQUÍ
        pass

    def floor(self, key):
        """
        Mayor clave ≤ key.
        Retornar None si todas las claves son mayores que key.
        """
        node = self._floor(self._root, key)
        return None if node is None else node.key

    def _floor(self, node, key):
        """
        HINT:
          si node is None → None
          si key == node.key → node
          si key < node.key → _floor(node.left, key)
          si key > node.key → t = _floor(node.right, key)
                              retornar t si t no es None, si no retornar node
        """
        # TU CÓDIGO AQUÍ
        pass

    def ceiling(self, key):
        """
        Menor clave ≥ key.
        Retornar None si todas las claves son menores que key.
        """
        node = self._ceiling(self._root, key)
        return None if node is None else node.key

    def _ceiling(self, node, key):
        """
        HINT: simétrico a _floor
          si key > node.key → _ceiling(node.right, key)
          si key < node.key → t = _ceiling(node.left, key)
                              retornar t si t no es None, si no retornar node
        """
        # TU CÓDIGO AQUÍ
        pass

    def rank(self, key):
        """Número de claves menores que key."""
        return self._rank(self._root, key)

    def _rank(self, node, key):
        """
        HINT:
          si node is None → 0
          si key < node.key → _rank(node.left, key)
          si key > node.key → 1 + _size(node.left) + _rank(node.right, key)
          si key == node.key → _size(node.left)
        """
        # TU CÓDIGO AQUÍ
        pass

    def select(self, k):
        """Clave de rango k (0-based)."""
        if k < 0 or k >= self.size():
            raise IndexError(f"rango {k} inválido")
        return self._select(self._root, k).key

    def _select(self, node, k):
        """
        HINT:
          t = _size(node.left)
          si t > k → _select(node.left, k)
          si t < k → _select(node.right, k - t - 1)
          si t == k → node
        """
        # TU CÓDIGO AQUÍ
        pass

    def keys_range(self, lo, hi):
        """Claves en [lo, hi] en orden."""
        result = []
        self._keys_range(self._root, lo, hi, result)
        return result

    def _keys_range(self, node, lo, hi, result):
        """
        HINT: recorrido inorden con poda:
          si lo < node.key → explorar izquierda
          si lo <= node.key <= hi → agregar
          si hi > node.key → explorar derecha
        """
        # TU CÓDIGO AQUÍ
        pass

In [ ]:
# ══════════════════════════════════════════
# VERIFICACIÓN AUTOMÁTICA — NO MODIFICAR
# ══════════════════════════════════════════

class TestOrderedBST(unittest.TestCase):
    """Operaciones de orden sobre el BST de S E X A R H M."""

    def setUp(self):
        self.obst = OrderedBST()
        for k, v in [('S', 0), ('E', 1), ('X', 7), ('A', 8), ('R', 3), ('H', 5), ('M', 9)]:
            self.obst.put(k, v)

    def test_01_min(self):
        """min() = 'A'"""
        self.assertEqual(self.obst.min(), 'A')

    def test_02_max(self):
        """max() = 'X'"""
        self.assertEqual(self.obst.max(), 'X')

    def test_03_floor_existente(self):
        """floor('H') = 'H'"""
        self.assertEqual(self.obst.floor('H'), 'H')

    def test_04_floor_intermedia(self):
        """floor('G') = 'E'"""
        self.assertEqual(self.obst.floor('G'), 'E')

    def test_05_floor_mayor_que_max(self):
        """floor('Z') = 'X'"""
        self.assertEqual(self.obst.floor('Z'), 'X')

    def test_06_floor_menor_que_min(self):
        """floor('0') = None"""
        self.assertIsNone(self.obst.floor('0'))

    def test_07_ceiling_existente(self):
        """ceiling('R') = 'R'"""
        self.assertEqual(self.obst.ceiling('R'), 'R')

    def test_08_ceiling_intermedia(self):
        """ceiling('G') = 'H'"""
        self.assertEqual(self.obst.ceiling('G'), 'H')

    def test_09_ceiling_mayor_que_max(self):
        """ceiling('Z') = None"""
        self.assertIsNone(self.obst.ceiling('Z'))

    def test_10_ceiling_menor_que_min(self):
        """ceiling('0') = 'A'"""
        self.assertEqual(self.obst.ceiling('0'), 'A')

    def test_11_rank_A(self):
        """rank('A') = 0"""
        self.assertEqual(self.obst.rank('A'), 0)

    def test_12_rank_E(self):
        """rank('E') = 1"""
        self.assertEqual(self.obst.rank('E'), 1)

    def test_13_rank_H(self):
        """rank('H') = 2"""
        self.assertEqual(self.obst.rank('H'), 2)

    def test_14_rank_X(self):
        """rank('X') = 6"""
        self.assertEqual(self.obst.rank('X'), 6)

    def test_15_rank_ausente(self):
        """rank('G') = 2 (clave ausente)"""
        self.assertEqual(self.obst.rank('G'), 2)

    def test_16_select_0(self):
        """select(0) = 'A'"""
        self.assertEqual(self.obst.select(0), 'A')

    def test_17_select_2(self):
        """select(2) = 'H'"""
        self.assertEqual(self.obst.select(2), 'H')

    def test_18_select_6(self):
        """select(6) = 'X'"""
        self.assertEqual(self.obst.select(6), 'X')

    def test_19_keys_range_E_R(self):
        """keys_range('E', 'R') = E H M R"""
        self.assertEqual(self.obst.keys_range('E', 'R'), ['E', 'H', 'M', 'R'])

    def test_20_keys_range_F_N(self):
        """keys_range('F', 'N') = H M"""
        self.assertEqual(self.obst.keys_range('F', 'N'), ['H', 'M'])

    def test_21_keys_range_vacio(self):
        """keys_range('T', 'W') = []"""
        self.assertEqual(self.obst.keys_range('T', 'W'), [])


correr_pruebas("2B", TestOrderedBST)

## PARTE 2C: Aplicación — Horario de Trenes (10 minutos)

Usa tu `OrderedBST` con el mismo horario de la clase anterior y responde las mismas consultas.  
Compara los resultados para verificar que ambas implementaciones (array vs árbol) dan los mismos resultados.

In [ ]:
# Datos del horario — NO MODIFICAR
HORARIO = [
    ('08:30', 'Alameda'), ('06:10', 'Constitución'), ('10:00', 'Maipu'),
    ('07:45', 'Lo Espejo'), ('13:00', 'San Bernardo'), ('09:15', 'Pudahuel'),
    ('15:20', 'Buin'), ('11:30', 'Cerrillos'),
]
# Nota: insertamos en orden NO secuencial a propósito para generar un árbol balanceado
print("Orden de inserción:", [h for h,_ in HORARIO])

In [ ]:
# TODO: Carga el horario y responde las consultas con tu OrderedBST

horario = OrderedBST()

# 1. Cargar el horario
# TU CÓDIGO AQUÍ

# Visualizar el árbol resultante
dibujar_bst(horario._root, "BST del horario de trenes")
print(f"Inorden: {horario.keys()}")
print(f"Altura del árbol: {altura_bst(horario._root)}")
print()

# 2. Primer y último tren
primer_tren = None   # TU CÓDIGO AQUÍ
ultimo_tren = None   # TU CÓDIGO AQUÍ
print(f"Primer tren: {primer_tren} → {horario.get(primer_tren)}")
print(f"Último tren: {ultimo_tren} → {horario.get(ultimo_tren)}")

# 3. floor y ceiling de '09:00'
antes_9 = None   # TU CÓDIGO AQUÍ
desde_9 = None   # TU CÓDIGO AQUÍ
print(f"Último antes de 09:00: {antes_9} → {horario.get(antes_9)}")
print(f"Próximo desde 09:00:   {desde_9} → {horario.get(desde_9)}")

# 4. rank de '10:00'
n_antes_10 = None   # TU CÓDIGO AQUÍ
print(f"Trenes antes de 10:00: {n_antes_10}")

# 5. 4to tren (select, 0-based)
cuarto_tren = None   # TU CÓDIGO AQUÍ
print(f"4to tren (select(3)): {cuarto_tren} → {horario.get(cuarto_tren)}")

# 6. Trenes entre 08:00 y 12:00
trenes_manana = None   # TU CÓDIGO AQUÍ
print(f"Trenes 08:00-12:00: {trenes_manana}")

In [ ]:
# ══════════════════════════════════════════
# VERIFICACIÓN AUTOMÁTICA — NO MODIFICAR
# ══════════════════════════════════════════

class TestHorarioBST(unittest.TestCase):
    """Respuestas del horario de trenes con OrderedBST."""

    def test_1_size(self):
        """el horario tiene 8 trenes"""
        self.assertEqual(horario.size(), 8)

    def test_2_primer_tren(self):
        """primer tren (min) = '06:10'"""
        self.assertEqual(primer_tren, '06:10')

    def test_3_ultimo_tren(self):
        """último tren (max) = '15:20'"""
        self.assertEqual(ultimo_tren, '15:20')

    def test_4_antes_de_las_9(self):
        """floor('09:00') = '08:30'"""
        self.assertEqual(antes_9, '08:30')

    def test_5_desde_las_9(self):
        """ceiling('09:00') = '09:15'"""
        self.assertEqual(desde_9, '09:15')

    def test_6_antes_de_las_10(self):
        """rank('10:00') = 4"""
        self.assertEqual(n_antes_10, 4)

    def test_7_cuarto_tren(self):
        """select(3) = '09:15'"""
        self.assertEqual(cuarto_tren, '09:15')

    def test_8_trenes_de_la_manana(self):
        """keys_range('08:00', '12:00')"""
        self.assertEqual(trenes_manana, ['08:30', '09:15', '10:00', '11:30'])


correr_pruebas("2C", TestHorarioBST)

## PARTE 2D: Análisis Visual — Forma del Árbol (extra)

Compara visualmente la forma del árbol con inserción aleatoria vs inserción ordenada.

In [ ]:
# Genera y compara dos árboles con los mismos 15 números
claves = list(range(1, 16))

# Árbol con inserción aleatoria
random.shuffle(claves)
bst_rand = OrderedBST()
for k in claves: bst_rand.put(k, k)

# Árbol con inserción ordenada (peor caso)
bst_sorted = OrderedBST()
for k in sorted(claves): bst_sorted.put(k, k)

print(f"Inserción aleatoria: orden = {claves}")
print(f"Altura: {altura_bst(bst_rand._root)}  (referencia: 2·log₂(15) ≈ {2*math.log2(15):.1f})")
dibujar_bst(bst_rand._root, f"Árbol aleatorio — altura = {altura_bst(bst_rand._root)}")

print(f"\nInserción ordenada: orden = {sorted(claves)}")
print(f"Altura: {altura_bst(bst_sorted._root)}  (peor caso = N = 15)")
dibujar_bst(bst_sorted._root, f"Árbol ordenado (degenerado) — altura = {altura_bst(bst_sorted._root)}")

### Preguntas de Análisis (edita esta celda)

**1.** ¿Cuántas veces más alto es el árbol degenerado que el árbol aleatorio para N=15?  
**Respuesta:**

---

**2.** Con el árbol del horario de trenes (8 entradas, insertadas en orden no secuencial), ¿cuál es la altura? ¿Coincide con ~2·log₂(8)?  
**Respuesta:**

---

**3.** Si el horario se insertara en orden cronológico (06:10, 07:45, 08:30, ...), ¿qué forma tendría el árbol y qué altura?  
**Respuesta:**

In [ ]:
# ══════════════════════════════════════════
# PUNTAJE FINAL — NO MODIFICAR
# ══════════════════════════════════════════
obtenidos = sum(o for o, _ in PUNTAJE.values())
posibles = sum(p for _, p in PUNTAJE.values())
print(f"\n{'='*50}")
for parte, (o, p) in PUNTAJE.items():
    print(f"  Parte {parte}: {o}/{p}")
print(f"  PUNTAJE TOTAL: {obtenidos}/{posibles} puntos")
print(f"{'='*50}")
porcentaje = obtenidos / posibles * 100 if posibles else 0
nota = 1.0 + (obtenidos / posibles) * 6.0 if posibles else 1.0
print(f"  Porcentaje: {porcentaje:.1f}%")
print(f"  Nota estimada (escala 1-7): {nota:.1f}")